## 1. State the project vision and question

**Status: Done**

**Project vision:** Build a reliable cleaned stock-price dataset for the Financial Agent project.

**Business question:** How do stock prices and trading activity differ across stock symbols and over time?

**Main metrics:** adjusted closing price, price change, and trading volume.

In [1]:
import pandas as pd

prices = pd.read_parquet("Data/prices.parquet")

prices.head()

,id,symbol,date,open,high,low,close,adj_close,volume
0,1,AAPL,2024-12-02,237.270004,240.789993,237.160004,239.589996,238.288300,48137100
1,2,AAPL,2024-12-03,239.809998,242.759995,238.899994,242.649994,241.331665,38861000
2,3,AAPL,2024-12-04,242.869995,244.110001,241.250000,243.009995,241.689728,44383900
3,4,AAPL,2024-12-05,243.990005,244.539993,242.130005,243.039993,241.719559,40033900
4,5,AAPL,2024-12-06,242.910004,244.630005,242.080002,242.839996,241.520630,36870600


In [2]:
prices.info()

<class 'pandas.DataFrame'>
RangeIndex: 2656 entries, 0 to 2655
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         2656 non-null   int32  
 1   symbol     2656 non-null   str    
 2   date       2656 non-null   object 
 3   open       2656 non-null   float64
 4   high       2656 non-null   float64
 5   low        2656 non-null   float64
 6   close      2656 non-null   float64
 7   adj_close  2656 non-null   float64
 8   volume     2656 non-null   int64  
dtypes: float64(5), int32(1), int64(1), object(1), str(1)
memory usage: 187.2+ KB


In [3]:
import pandas as pd

prices = pd.read_parquet("Data/prices.parquet").copy()
prices["symbol"] = prices["symbol"].astype(str).str.strip()
prices["date"] = pd.to_datetime(prices["date"], errors="coerce")

for col in ["open", "high", "low", "close", "adj_close", "volume"]:
    prices[col] = pd.to_numeric(prices[col], errors="coerce")

prices = prices.dropna(subset=["symbol", "date", "open", "high", "low", "close", "adj_close", "volume"]).copy()
prices = prices.drop_duplicates()
prices = prices.sort_values(["symbol", "date"]).reset_index(drop=True)

invalid_ohlc = prices[
    (prices["high"] < prices[["open", "close"]].max(axis=1)) |
    (prices["low"] > prices[["open", "close"]].min(axis=1)) |
    (prices["volume"] < 0)
]

if not invalid_ohlc.empty:
    prices = prices.drop(index=invalid_ohlc.index)

prices.head()

,id,symbol,date,open,high,low,close,adj_close,volume
0,1,AAPL,2024-12-02,237.270004,240.789993,237.160004,239.589996,238.288300,48137100
1,2,AAPL,2024-12-03,239.809998,242.759995,238.899994,242.649994,241.331665,38861000
2,3,AAPL,2024-12-04,242.869995,244.110001,241.250000,243.009995,241.689728,44383900
3,4,AAPL,2024-12-05,243.990005,244.539993,242.130005,243.039993,241.719559,40033900
4,5,AAPL,2024-12-06,242.910004,244.630005,242.080002,242.839996,241.520630,36870600


In [4]:
prices.info()
print(f"Rows after cleaning: {len(prices)}")
print("Missing values:\n", prices.isna().sum())
print(f"Duplicate rows: {prices.duplicated().sum()}")
invalid_ohlc_count = int(
    ((prices["high"] < prices[["open", "close"]].max(axis=1)) |
     (prices["low"] > prices[["open", "close"]].min(axis=1)) |
     (prices["volume"] < 0)).sum()
)
print(f"Invalid OHLC/volume rows: {invalid_ohlc_count}")
prices.to_parquet("Data/prices_cleaned.parquet", index=False)
print("Cleaned file saved to Data/prices_cleaned.parquet")

<class 'pandas.DataFrame'>
RangeIndex: 2656 entries, 0 to 2655
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype        
---  ------     --------------  -----        
 0   id         2656 non-null   int32        
 1   symbol     2656 non-null   str          
 2   date       2656 non-null   datetime64[s]
 3   open       2656 non-null   float64      
 4   high       2656 non-null   float64      
 5   low        2656 non-null   float64      
 6   close      2656 non-null   float64      
 7   adj_close  2656 non-null   float64      
 8   volume     2656 non-null   int64        
dtypes: datetime64[s](1), float64(5), int32(1), int64(1), str(1)
memory usage: 187.5 KB
Rows after cleaning: 2656
Missing values:
 id           0
symbol       0
date         0
open         0
high         0
low          0
close        0
adj_close    0
volume       0
dtype: int64
Duplicate rows: 0
Invalid OHLC/volume rows: 0
Cleaned file saved to Data/prices_cleaned.parquet


In [5]:
original = pd.read_parquet("Data/prices.parquet").copy()

print("========== 2. DEFINE ONE ROW ==========")
print("One row = one stock symbol on one trading date")
print("Original rows:", len(original))
print("Unique symbol-date combinations:",
      original[["symbol", "date"]].drop_duplicates().shape[0])
print("Repeated symbol-date keys:",
      original.duplicated(subset=["symbol", "date"]).sum())


print("\n========== 3. PRESERVE SOURCE ==========")
print("Source: TheFinAI/Herculean - prices subset")
print("Original file: Data/prices.parquet")
print("Cleaned file: Data/prices_cleaned.parquet")
print("Original file was read only and not overwritten.")
print("Download date: October 5, 2026")
print('Load command: pd.read_parquet("Data/prices.parquet")')

print("\n========== 4. COLUMNS AND TYPES ==========")
print("ORIGINAL TYPES:")
print(original.dtypes)
print("\nCLEANED TYPES:")
print(prices.dtypes)

date_failed = (
    pd.to_datetime(original["date"], errors="coerce").isna()
    & original["date"].notna()
).sum()

numeric_failed = {}
for col in ["open", "high", "low", "close", "adj_close", "volume"]:
    converted = pd.to_numeric(original[col], errors="coerce")
    numeric_failed[col] = int(
        (converted.isna() & original[col].notna()).sum()
    )

print("\nFailed date conversions:", date_failed)
print("Failed numeric conversions:", numeric_failed)


print("\n========== 5. STANDARDIZE TEXT ==========")
print("Symbol examples BEFORE:")
print(original["symbol"].head().tolist())

clean_symbols = original["symbol"].astype(str).str.strip()

print("Symbol examples AFTER:")
print(clean_symbols.head().tolist())

spaces_before = (
    original["symbol"].astype(str)
    != original["symbol"].astype(str).str.strip()
).sum()

print("Symbols changed by trimming whitespace:", spaces_before)


print("\n========== 6. DUPLICATES ==========")
print("Exact duplicate rows BEFORE:",
      original.duplicated().sum())

print("Repeated IDs BEFORE:",
      original["id"].duplicated().sum())

print("Repeated symbol-date keys BEFORE:",
      original.duplicated(subset=["symbol", "date"]).sum())

print("Exact duplicate rows AFTER:",
      prices.duplicated().sum())

print("Repeated IDs AFTER:",
      prices["id"].duplicated().sum())

print("Repeated symbol-date keys AFTER:",
      prices.duplicated(subset=["symbol", "date"]).sum())


print("\n========== 7. MISSING DATA ==========")
print("Missing values BEFORE:")
print(original.isna().sum())

print("\nMissing values AFTER:")
print(prices.isna().sum())


print("\n========== 8. SUSPICIOUS VALUES ==========")

original_check = original.copy()
original_check["date_check"] = pd.to_datetime(
    original_check["date"], errors="coerce"
)

invalid_dates = original_check["date_check"].isna().sum()

negative_prices = (
    (original_check[["open", "high", "low", "close", "adj_close"]] < 0)
    .any(axis=1)
    .sum()
)

negative_volume = (original_check["volume"] < 0).sum()

invalid_high_low = (
    original_check["high"] < original_check["low"]
).sum()

invalid_ohlc_before = (
    (original_check["high"] <
     original_check[["open", "close"]].max(axis=1))
    |
    (original_check["low"] >
     original_check[["open", "close"]].min(axis=1))
).sum()

print("Invalid dates:", invalid_dates)
print("Rows with negative prices:", negative_prices)
print("Rows with negative volume:", negative_volume)
print("Rows where high < low:", invalid_high_low)
print("Invalid OHLC relationships:", invalid_ohlc_before)


print("\n========== 9. MERGES ==========")
print("Not applicable - only the prices subset was used and no tables were merged.")


print("\n========== 10. RECONCILE CHANGES ==========")
print("Starting rows:", len(original))
print("Final rows:", len(prices))
print("Rows removed:", len(original) - len(prices))

print("Starting total volume:", original["volume"].sum())
print("Final total volume:", prices["volume"].sum())


print("\n========== 11. CHECK CALCULATIONS ==========")

print("Records used for close-price mean:",
      prices["close"].notna().sum())

print("Average close price:",
      prices["close"].mean())

print("Records used for volume total:",
      prices["volume"].notna().sum())

print("Total volume:",
      prices["volume"].sum())

print("Missing symbol group keys:",
      prices["symbol"].isna().sum())


print("========== 12. VERIFY FIVE INDIVIDUAL RECORDS ==========")

# Choose five records, including boundary and unusual cases
candidate_checks = [
    (original.index[0], "First record in the dataset"),
    (original.index[-1], "Last record in the dataset"),
    (original["volume"].idxmax(), "Highest trading volume"),
    (original["close"].idxmin(), "Lowest closing price"),
    (original["close"].idxmax(), "Highest closing price")
]

selected = []
used_ids = set()

for idx, reason in candidate_checks:
    record_id = original.loc[idx, "id"]

    if record_id not in used_ids:
        selected.append((idx, reason))
        used_ids.add(record_id)

# Fill to 5 records if any selections overlap
for idx in original.index:
    if len(selected) >= 5:
        break

    record_id = original.loc[idx, "id"]

    if record_id not in used_ids:
        selected.append((idx, "Additional record for verification"))
        used_ids.add(record_id)


for number, (idx, reason) in enumerate(selected[:5], start=1):

    orig = original.loc[idx]

    # Expected result BEFORE checking cleaned output
    expected_symbol = str(orig["symbol"]).strip()
    expected_date = pd.to_datetime(orig["date"], errors="coerce")
    expected_close = orig["close"]
    expected_volume = orig["volume"]

    print(f"\n--- RECORD CHECK {number} ---")
    print("Reason selected:", reason)

    print(
        "ORIGINAL:",
        "id =", orig["id"],
        "| symbol =", orig["symbol"],
        "| date =", orig["date"],
        "| close =", orig["close"],
        "| volume =", orig["volume"]
    )

    print(
        "EXPECTED:",
        "symbol =", expected_symbol,
        "| date =", expected_date,
        "| close =", expected_close,
        "| volume =", expected_volume
    )

    # Now check actual cleaned result
    actual = prices[prices["id"] == orig["id"]]

    if len(actual) == 1:

        a = actual.iloc[0]

        print(
            "ACTUAL:",
            "symbol =", a["symbol"],
            "| date =", a["date"],
            "| close =", a["close"],
            "| volume =", a["volume"]
        )

        match = (
            a["symbol"] == expected_symbol
            and pd.to_datetime(a["date"]) == expected_date
            and a["close"] == expected_close
            and a["volume"] == expected_volume
        )

        print("MATCH:", match)

    elif len(actual) == 0:
        print("ACTUAL: Record was removed during cleaning")
        print("MATCH: False")

    else:
        print("ACTUAL: Record appears more than once")
        print("MATCH: False")

========== 2. DEFINE ONE ROW ==========
One row = one stock symbol on one trading date
Original rows: 2656
Unique symbol-date combinations: 2656
Repeated symbol-date keys: 0

========== 3. PRESERVE SOURCE ==========
Source: TheFinAI/Herculean - prices subset
Original file: Data/prices.parquet
Cleaned file: Data/prices_cleaned.parquet
Original file was read only and not overwritten.
Download date: October 5, 2026
Load command: pd.read_parquet("Data/prices.parquet")

========== 4. COLUMNS AND TYPES ==========
ORIGINAL TYPES:
id             int32
symbol           str
date          object
open         float64
high         float64
low          float64
close        float64
adj_close    float64
volume         int64
dtype: object

CLEANED TYPES:
id                   int32
symbol                 str
date         datetime64[s]
open               float64
high               float64
low                float64
close              float64
adj_close          float64
volume               int64
dtype: ob

## 13. Test reproducibility

**Status: Done**

I restarted the Jupyter kernel and used Run All. The notebook loads the original `Data/prices.parquet`, applies the cleaning rules, runs the validation checks, and recreates `Data/prices_cleaned.parquet`.

Dependencies: Python, pandas, pyarrow.

## 14. Document remaining limitations

**Status: Done**

This cleaning work uses only the prices subset of the Herculean dataset. The cleaned values were not independently verified against an external stock-price provider. Unusually large prices or trading volumes were reviewed but were not automatically removed because unusual values are not necessarily incorrect. No tables were merged.